# Partie A — Accessibilité en transports publics par commune

**Objectif** : pour chaque commune de domicile, mesurer à quel point les transports publics (TP) sont une alternative réaliste à la voiture pour arriver au travail vers 8h00.

**Entrée** : `parking_data.csv` 
**Sorties** : `transit_access.csv` (une ligne par commune), `table_A1.csv` (Table A1), `transit_cache.json` (archive des réponses API : relancer donne les mêmes résultats)

**Sources** : transport.opendata.ch (horaires officiels CFF), api3.geo.admin.ch (swisstopo : communes et registre des bâtiments RegBL), Google Maps (temps voiture).

Exécuter les cellules dans l'ordre (*Run All*).

## 0. Librairies

In [2]:
import json, math, os, re, time, unicodedata
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from threading import RLock

import numpy as np
import pandas as pd
import requests

## 0' Paramètres

Tous les choix de méthode sont réglés dans la cellule ci-dessous.

**Horaire**
- **Date de référence** : jeudi 10.12.2026, dernier jour ouvrable de l'horaire 2026. L'API ne fournit pas les horaires passés (2024-2025).
- **Destination** : gare principale du site (Lausanne ou Sion), l'adresse exacte des sites n'étant pas connue.
- **Services exclus** : lignes apparues après l'enquête 2024-2025 (ligne 9 prolongée vers Renens/Crissier, ligne 54, tram T1).

**Choix de l'arrêt de départ de chaque commune**
1. *Logements* : tous les bâtiments d'habitation de la commune (registre fédéral RegBL), chaque bâtiment pesant son nombre de logements.
2. *Arrêts candidats* : arrêts proches du point moyen des logements et du centre de chaque quart de la commune (pour ne pas oublier une gare en bordure), + l'arrêt principal de la commune (`MAIN_STOP`). Seuls les arrêts avec au moins un départ entre 6h et 9h sont gardés.
3. *Temps total moyen* de chaque arrêt = marche moyenne depuis tous les logements jusqu'à l'arrêt + trajet TP le plus rapide jusqu'à la gare du site, arrivée entre 7h30 et 8h00.
4. *Arrêt retenu* : celui avec le plus petit temps total moyen. Si plusieurs arrêts sont à moins de `TIE_MIN` = 1 minute de ce minimum (l'horaire est à la minute près), on garde celui dont la marche moyenne est la plus courte.

**Marche et voiture**
- **Marche** : distance à vol d'oiseau × `WALK_DETOUR` = 1,3 (détour par les rues), à `WALK_M_PER_MIN` = 75 m/min (4,5 km/h).
- **Voiture** : temps Google Maps (`CAR_TIMES`, du point moyen des logements à la gare, arrivée 8h00 un jeudi), milieu de la fourchette + `CAR_TERMINAL_MIN` = 5 min pour se garer et marcher jusqu'au site.

In [3]:
REF_DATE = "2026-12-10"     # jeudi, dernier jour ouvrable de l'horaire 2026 avant le
                            # changement du 13.12.2026 (pas encore de tram T1)
WALK_DETOUR = 1.3           # chemin à pied = 1.3 x distance à vol d'oiseau
WALK_M_PER_MIN = 75         # 4.5 km/h
CAR_TERMINAL_MIN = 5        # se garer + marcher jusqu'au site
TIE_MIN = 1.0               # arrêts à moins d'1 min d'écart = égalité -> marche la plus courte

SITE_STOP = {"Lausanne": "Lausanne", "Sion": "Sion"}           # destination = gare principale
SITE_COORD = {"Lausanne": (46.5167, 6.6291), "Sion": (46.2274, 7.3592)}

# Arrêt principal de chaque localité (toujours ajouté aux arrêts candidats)
MAIN_STOP = {
    "BOTTENS": "Bottens VD", "BUSSIGNY": "Bussigny", "CHAVANNES PRES RENENS": "Chavannes-près-Renens",
    "CHESEAUX SUR LAUSANNE": "Cheseaux", "CORCELLES LE JORAT": "Corcelles-le-Jorat",
    "CRISSIER": "Crissier, centre", "CUGY": "Cugy VD", "ECUBLENS": "Ecublens VD",
    "EPALINGES": "Epalinges, Croisettes", "FROIDEVILLE": "Froideville", "LAUSANNE": "Lausanne, Chauderon",
    "LE MONT SUR LAUSANNE": "Le Mont-sur-Lausanne", "LUTRY": "Lutry", "MEZIERES": "Mézières VD",
    "MONTPREVEYRES": "Montpreveyres", "MORGES": "Morges", "PRILLY": "Prilly", "PULLY": "Pully",
    "RENENS": "Renens VD", "ROMANEL SUR LAUSANNE": "Romanel-sur-Lausanne", "ROPRAZ": "Ropraz",
    "SAVIGNY": "Savigny VD", "SERVION": "Servion", "VULLIENS": "Vulliens", "ARBAZ": "Arbaz",
    "ARDON": "Ardon", "AYENT": "Ayent", "BRAMOIS": "Bramois", "CONTHEY": "Conthey",
    "EVOLENE": "Evolène", "GRIMISUAT": "Grimisuat", "HEREMENCE": "Hérémence", "NENDAZ": "Basse-Nendaz",
    "SAINT MARTIN": "Saint-Martin VS", "SALINS": "Salins VS", "SAVIESE": "Savièse",
    "SION": "Sion, Platta", "VETROZ": "Vétroz", "VEX": "Vex",
}
NPA_ONLY = {"LAUSANNE", "BRAMOIS", "SALINS"}  # logements pris par code postal (quartier / ancien village)   

# Services introduits APRÈS l'enquête 2024-2025 -> exclus
#   ligne 9 prolongée vers Renens/Crissier (08.2026), ligne 54 (12.2025), tram T1
NEW_SERVICES = {("B", "9"): ["Crissier", "Renens"], ("B", "54"): None,
                ("T", "1"): None, ("T", "T1"): None}

# Temps voiture Google Maps (min), du point moyen des logements à la gare du site,
# arrivée 08:00 un jeudi : (borne basse, borne haute) de la fourchette affichée
CAR_TIMES = {
    "ARBAZ": (20, 30), "ARDON": (14, 22), "AYENT": (20, 20), "BOTTENS": (24, 45), "BRAMOIS": (9, 12),
    "BUSSIGNY": (14, 24), "CHAVANNES PRES RENENS": (12, 20), "CHESEAUX SUR LAUSANNE": (18, 35),
    "CONTHEY": (8, 14), "CORCELLES LE JORAT": (26, 50), "CRISSIER": (16, 28), "CUGY": (22, 40),
    "ECUBLENS": (14, 24), "EPALINGES": (14, 35), "EVOLENE": (35, 35), "FROIDEVILLE": (24, 45),
    "GRIMISUAT": (12, 20), "HEREMENCE": (22, 30), "LAUSANNE": (8, 16), "LE MONT SUR LAUSANNE": (14, 28),
    "LUTRY": (12, 22), "MEZIERES": (26, 45), "MONTPREVEYRES": (24, 45), "MORGES": (16, 24),
    "NENDAZ": (26, 30), "PRILLY": (9, 16), "PULLY": (7, 16), "RENENS": (12, 22),
    "ROMANEL SUR LAUSANNE": (12, 24), "ROPRAZ": (26, 50), "SAINT MARTIN": (31, 31), "SALINS": (14, 14),
    "SAVIESE": (16, 24), "SAVIGNY": (18, 35), "SERVION": (26, 45), "SION": (4, 4), "VETROZ": (12, 18),
    "VEX": (14, 18), "VULLIENS": (30, 55),
}

## 1. Nettoyage des données brutes

## 1. Données brutes : observation et nettoyage

On regarde d'abord les données telles qu'elles sont fournies (1a à 1e), pour repérer les données manquantes, les valeurs impossibles et les incohérences. On corrige ensuite les communes (1f), puis le reste (1g), et on agrège par commune (1h). Le fichier `parking_data.csv` n'est jamais modifié.

### 1a. Vue d'ensemble

In [4]:
raw = pd.read_csv("parking_data.csv")
print(raw.shape)
print("Employés :", raw["employee_id"].nunique(), "| années :", sorted(raw["year"].unique()))
print("Lignes par employé :", raw.groupby("employee_id").size().value_counts().to_dict())
print("Employés par site :", raw.drop_duplicates("employee_id")["site"].value_counts().to_dict())
raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'parking_data.csv'

### 1b. Données manquantes et doublons

In [ ]:
print("Valeurs manquantes par colonne :")
print(raw.isna().sum()[raw.isna().sum() > 0])
print("\nDoublons (même employé, même année) :", raw.duplicated(["employee_id", "year"]).sum())
miss = raw[raw["gross_income_chf"].isna()]
n_years = miss.groupby("employee_id").size()
print("Revenu manquant :", miss["employee_id"].nunique(), "employés, dont",
      (n_years == 1).sum(), "pour une seule année et", (n_years == 2).sum(), "pour les deux années")

Valeurs manquantes par colonne :
gross_income_chf    110
dtype: int64

Doublons (même employé, même année) : 0
Revenu manquant : 106 employés, dont 102 pour une seule année et 4 pour les deux années


### 1c. Cohérence entre 2024 et 2025

Ces caractéristiques ne devraient pas changer d'une année à l'autre pour un même employé.

In [ ]:
for col in ["site", "home_commune", "home_postcode", "distance_km", "age",
            "n_children", "fte", "gross_income_chf"]:
    n = (raw.groupby("employee_id")[col].nunique() > 1).sum()
    print(f"{col:<18} change pour {n} employés")

site               change pour 0 employés
home_commune       change pour 45 employés
home_postcode      change pour 0 employés
distance_km        change pour 0 employés
age                change pour 0 employés
n_children         change pour 0 employés
fte                change pour 0 employés
gross_income_chf   change pour 0 employés


### 1d. Plages de valeurs et cohérence logique

On compare aux valeurs possibles d'après la description des variables (ex. `car_days_per_week` entre 0 et 5).

In [ ]:
print(raw.describe().T[["min", "max", "mean"]])
print("\ncar_days_per_week > 5 :")
print(raw.loc[raw["car_days_per_week"] > 5, ["employee_id", "year", "main_mode", "car_days_per_week"]])
print("\nuses_car incohérent avec main_mode :", ((raw["main_mode"] == "car") != (raw["uses_car"] == 1)).sum())
print("Non-automobilistes avec des jours en voiture :", ((raw["uses_car"] == 0) & (raw["car_days_per_week"] > 0)).sum())
print("\nModes :", raw["main_mode"].value_counts().to_dict())
print("\nTarif de parking par site et année :")
print(raw.groupby(["site", "year"])["parking_fee_chf"].unique())

                       min       max          mean
year                2024.0    2025.0   2024.500000
parking_fee_chf        2.0       4.0      2.800000
uses_car               0.0       1.0      0.511000
car_days_per_week      0.0       7.4      1.815650
home_postcode       1004.0    1997.0   1224.999000
distance_km            1.3      37.5      8.343900
age                   22.0      64.0     42.567000
n_children             0.0       2.0      0.748000
fte                    0.6       1.0      0.962600
gross_income_chf   24300.0  222500.0  94784.338624

car_days_per_week > 5 :
  employee_id  year main_mode  car_days_per_week
3       E1001  2025       car                7.4

uses_car incohérent avec main_mode : 0
Non-automobilistes avec des jours en voiture : 0

Modes : {'car': 1022, 'bus': 705, 'bike': 273}

Tarif de parking par site et année :
site      year
Lausanne  2024    [2.0]
          2025    [4.0]
Sion      2024    [2.0]
          2025    [2.0]
Name: parking_fee_chf, dtype: 

### 1e. Communes et codes postaux

Les noms de communes sont écrits de plusieurs façons (accents, tirets, majuscules). On les harmonise pour voir les variantes, puis on vérifie qu'une commune n'a qu'un code postal et qu'il est le bon.

In [ ]:
def normalize(name):
    """Majuscules, sans accents ni ponctuation : 'Chavannes-près-Renens' -> 'CHAVANNES PRES RENENS'."""
    name = unicodedata.normalize("NFKD", str(name).upper().strip())
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]", " ", name)).strip()

check = raw.assign(commune_clean=raw["home_commune"].apply(normalize))
print("Orthographes différentes :", raw["home_commune"].nunique(), "-> communes après harmonisation :",
      check["commune_clean"].nunique())
pd.set_option("display.max_rows", 100)
check.groupby("commune_clean").agg(
    site=("site", "unique"),
    variantes=("home_commune", lambda s: sorted(s.unique())),
    npa=("home_postcode", "unique"),
    employes=("employee_id", "nunique"),
)

Orthographes différentes : 62 -> communes après harmonisation : 39


,site,variantes,npa,employes
commune_clean,,,,
ARBAZ,[Sion],[Arbaz],[1974],6
ARDON,[Sion],"[ARDON, Ardon]",[1957],11
AYENT,[Sion],"[AYENT, Ayent]",[1966],15
BOTTENS,[Lausanne],"[BOTTENS, Bottens]",[1041],15
BRAMOIS,[Sion],[Bramois],[1967],16
BUSSIGNY,[Lausanne],"[BUSSIGNY, Bussigny]",[1030],43
CHAVANNES PRES RENENS,[Lausanne],"[CHAVANNES PRES RENENS, Chavannes-pres-Renens]",[1022],20
CHESEAUX SUR LAUSANNE,[Lausanne],[Cheseaux-sur-Lausanne],[1033],21
CONTHEY,[Sion],[Conthey],[1964],20


**Ce que montre l'observation :**
- 62 orthographes pour 39 communes (accents, tirets, majuscules) → harmonisation.
- **Savièse** est écrite « Savieze » dans toutes les lignes (faute d'orthographe).
- **Salins** : les 8 employés ont le NPA **1994**, qui est celui d'Aproz (commune de Nendaz) ; le NPA de Salins est 1991.
- Chaque commune n'a qu'un seul NPA ; site, NPA, distance, âge, enfants, taux d'activité et revenu ne changent jamais entre 2024 et 2025.
- `home_commune` change entre 2024 et 2025 pour 45 employés, mais **uniquement par l'écriture** (ex. « FROIDEVILLE » / « Froideville ») : après harmonisation, aucun employé ne change de commune (vérifié en 1f).
- **Revenu** : seule variable avec des valeurs manquantes (110 lignes, 106 employés, dont 102 pour une seule année).
- **Valeur impossible** : E1001 déclare 7,4 jours de voiture par semaine en 2025 (maximum 5).
- `uses_car` correspond toujours à `main_mode`, et les tarifs correspondent aux consignes (Lausanne : 2 → 4 CHF ; Sion : 2 CHF).

### 1f. Correction des communes

- Noms harmonisés (`commune_clean`).
- « Savieze » → **SAVIESE**.
- Salins : on se réfère au nom de commune → NPA corrigé de 1994 en **1991**.

Les valeurs d'origine sont conservées (`commune_clean_raw`, `home_postcode_raw`).

In [ ]:
raw["commune_clean"] = raw["home_commune"].apply(normalize)
raw["commune_clean_raw"] = raw["commune_clean"]
raw["home_postcode_raw"] = raw["home_postcode"]

raw["commune_clean"] = raw["commune_clean"].replace({"SAVIEZE": "SAVIESE"})   # faute d'orthographe
raw.loc[raw["commune_clean"] == "SALINS", "home_postcode"] = 1991              # NPA 1994 = Aproz

print("Communes :", raw["commune_clean"].nunique())
print("Employés changeant de commune entre 2024 et 2025 (après harmonisation) :",
      (raw.groupby("employee_id")["commune_clean"].nunique() > 1).sum())
raw.loc[raw["commune_clean"].isin(["SAVIESE", "SALINS"]),
        ["commune_clean_raw", "commune_clean", "home_postcode_raw", "home_postcode"]].drop_duplicates()

Communes : 39
Employés changeant de commune entre 2024 et 2025 (après harmonisation) : 0


,commune_clean_raw,commune_clean,home_postcode_raw,home_postcode
1632,SAVIEZE,SAVIESE,1965,1965
1660,SALINS,SALINS,1994,1991


### 1g. Autres corrections

- **Revenu manquant** pour une seule année : on reprend la valeur de l'autre année (le revenu est constant par employé, cf. 1c). Les employés sans revenu les deux années restent manquants -> cas pour 4 employés soit (0.4%), il faudra les exclures de l'analyse de revenu ou remplacer pas une mediane des même profil (taux d'activité, age, lieu de travail)
- **Jours de voiture > 5** (impossible) : plafonnés à 5.

Valeurs d'origine conservées : `income_imputed` (revenu complété), `car_days_raw`. Ces corrections ne changent pas les résultats de la Partie A (qui n'utilise que les communes) mais servent aux parties B à E.

In [ ]:
raw["income_imputed"] = raw["gross_income_chf"].isna()
raw["gross_income_chf"] = raw.groupby("employee_id")["gross_income_chf"].transform(lambda s: s.fillna(s.mean()))
raw["income_imputed"] &= raw["gross_income_chf"].notna()
print("Revenus complétés :", raw["income_imputed"].sum(),
      "| encore manquants :", raw["gross_income_chf"].isna().sum(), "lignes")

raw["car_days_raw"] = raw["car_days_per_week"]
raw["car_days_per_week"] = raw["car_days_per_week"].clip(upper=5)
print("Jours de voiture plafonnés à 5 :", (raw["car_days_raw"] > 5).sum(), "ligne(s)")

Revenus complétés : 102 | encore manquants : 8 lignes
Jours de voiture plafonnés à 5 : 1 ligne(s)


### 1h. Une ligne par commune

In [ ]:
communes = (raw.groupby(["commune_clean", "site"])
               .agg(n_employees=("employee_id", "nunique"),
                    postcode=("home_postcode", "first"),
                    mean_distance_km=("distance_km", "mean"))
               .reset_index())
assert len(communes) == 39 and communes.n_employees.sum() == 1000
communes

,commune_clean,site,n_employees,postcode,mean_distance_km
0,ARBAZ,Sion,6,1974,11.483333
1,ARDON,Sion,11,1957,13.809091
2,AYENT,Sion,15,1966,12.773333
3,BOTTENS,Lausanne,15,1041,14.440000
4,BRAMOIS,Sion,16,1967,5.318750
5,BUSSIGNY,Lausanne,43,1030,9.788372
6,CHAVANNES PRES RENENS,Lausanne,20,1022,7.110000
7,CHESEAUX SUR LAUSANNE,Lausanne,21,1033,8.547619
8,CONTHEY,Sion,20,1964,7.225000
9,CORCELLES LE JORAT,Lausanne,13,1082,19.423077


## 2. Accès aux API

Chaque réponse est enregistrée dans `transit_cache.json` : une requête n'est faite qu'une seule fois, et relancer le notebook redonne exactement les mêmes résultats.
Les connexions qui utilisent un service introduit après 2025 sont écartées.

In [ ]:
CACHE_FILE = "transit_cache.json"
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
lock = RLock()


def save_cache():
    with lock:
        with open(CACHE_FILE + ".tmp", "w") as f:
            json.dump(cache, f)
        os.replace(CACHE_FILE + ".tmp", CACHE_FILE)


def get(url, params, transform=None):
    key = url + "?" + json.dumps(params, sort_keys=True)
    if key in cache:
        return cache[key]
    for attempt in range(4):                       # réessaie si l'API sature
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code != 429:
                break
        except requests.ConnectionError:
            pass
        time.sleep(5 * (attempt + 1))
    r.raise_for_status()
    data = transform(r.json()) if transform else r.json()
    with lock:
        cache[key] = data
        if len(cache) % 25 == 0:
            save_cache()
    time.sleep(0.2)
    return data


OPENDATA = "https://transport.opendata.ch/v1/"
GEO = "https://api3.geo.admin.ch/rest/services/api/"


def ts(s):                                         # '2026-12-10T07:52:00+0100' -> datetime
    return datetime.strptime(s[:19], "%Y-%m-%dT%H:%M:%S")


def at(hhmm):                                      # '08:00' -> datetime du jour de référence
    return datetime.strptime(f"{REF_DATE} {hhmm}", "%Y-%m-%d %H:%M")


def km(lat1, lon1, lat2, lon2):                    # distance à vol d'oiseau
    p = math.pi / 180
    a = (math.sin((lat2 - lat1) * p / 2) ** 2
         + math.cos(lat1 * p) * math.cos(lat2 * p) * math.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742 * math.asin(math.sqrt(a))


def is_new_service(category, number, boarding_stop):
    prefixes = NEW_SERVICES.get((category, str(number)), False)
    return prefixes is None or (prefixes and any(boarding_stop.startswith(p) for p in prefixes))


def connections(frm, to, hhmm, arrival, limit):
    """Connexions TP, sans celles qui utilisent un service introduit après 2025."""
    res = get(OPENDATA + "connections", {"from": frm, "to": to, "date": REF_DATE, "time": hhmm,
                                         "isArrivalTime": int(arrival), "limit": limit, "page": 0})
    ok = [c for c in res.get("connections", [])
          if not any(s.get("journey") and is_new_service(s["journey"].get("category"),
                                                         s["journey"].get("number"),
                                                         s["departure"]["station"]["name"] or "")
                     for s in c.get("sections", []))]
    return ok, res.get("connections", [])

## 3. Les trois indicateurs TP d'un arrêt

1. **Trajet de 8h** : le trajet le plus rapide qui arrive entre 7h30 et 8h00 (sinon le dernier avant 8h00).
2. **Départs 6h-9h** : nombre de départs distincts vers le site entre 6h et 9h.
3. **Première arrivée** : heure de la première arrivée possible au site le matin.

In [ ]:
def trip_0800(stop, site):
    """Trajet le plus rapide arrivant entre 07:30 et 08:00 (sinon le dernier avant 08:00)."""
    cons, _ = connections(stop, SITE_STOP[site], "08:00", arrival=True, limit=6)
    cons = [c for c in cons if ts(c["to"]["arrival"]) <= at("08:00")]
    pool = ([c for c in cons if ts(c["to"]["arrival"]) >= at("07:30")]
            or sorted(cons, key=lambda c: ts(c["to"]["arrival"]))[-1:])
    if not pool:
        return None
    return min(pool, key=lambda c: ts(c["to"]["arrival"]) - ts(c["from"]["departure"]))


def trip_minutes(c):
    return (ts(c["to"]["arrival"]) - ts(c["from"]["departure"])).total_seconds() / 60


def departures_6_9(stop, site):
    """Départs distincts 06:00-09:00 vers le site (l'API renvoie 16 connexions par requête)."""
    deps, hhmm = set(), "06:00"
    for _ in range(30):
        ok, allc = connections(stop, SITE_STOP[site], hhmm, arrival=False, limit=16)
        if not allc:
            break
        deps |= {ts(c["from"]["departure"]) for c in ok
                 if at("06:00") <= ts(c["from"]["departure"]) < at("09:00")}
        last = max(ts(c["from"]["departure"]) for c in allc)
        nxt = (last + timedelta(minutes=1)).strftime("%H:%M")
        if last >= at("09:00") or nxt <= hhmm:
            break
        hhmm = nxt
    return len(deps)


def first_arrival(stop, site):
    ok, _ = connections(stop, SITE_STOP[site], "04:00", arrival=False, limit=4)
    arr = [ts(c["to"]["arrival"]) for c in ok if ts(c["to"]["arrival"]).date() == at("04:00").date()]
    return min(arr).strftime("%H:%M") if arr else None


def served_6_9(stop_id, stop_name):
    """Au moins un départ (toutes directions) entre 06:00 et 09:00 à cet arrêt ?"""
    r = get(OPENDATA + "stationboard", {"id": stop_id, "datetime": f"{REF_DATE} 06:00", "limit": 200})
    return any(d.get("stop", {}).get("departure")
               and at("06:00") <= ts(d["stop"]["departure"]) < at("09:00")
               and not is_new_service(d.get("category"), d.get("number"), stop_name)
               for d in r.get("stationboard", []))

## 4. Logements et arrêt retenu d'une commune

1. **Logements** : tous les bâtiments d'habitation de la localité (registre fédéral RegBL), pondérés par leur nombre de logements. Lausanne : seulement le NPA des employés (1004).
2. **Arrêts candidats** : arrêts proches du point moyen des logements et du centre de chaque quart de la localité, + l'arrêt principal.
3. **Arrêt retenu** : parmi les arrêts desservis entre 6h et 9h, celui qui minimise **marche moyenne depuis tous les logements + trajet TP de 8h**. Égalité (< 1 min) → marche la plus courte.

In [ ]:
def commune_centre(commune, site, road_km):
    """Centre de la commune (swisstopo). Rejette les homonymes trop loin du site."""
    road_km = max(road_km, 10) if commune == site.upper() else road_km
    for layer in ("gg25", "gazetteer"):            # gg25 = communes ; gazetteer = villages
        r = get(GEO + "SearchServer", {"searchText": commune.title(), "type": "locations",
                                       "origins": layer, "sr": "4326"})
        for res in r.get("results", []):
            a = res["attrs"]
            label = re.sub(r"<[^>]+>", "", a.get("label", ""))
            if commune in normalize(label) and km(a["lat"], a["lon"], *SITE_COORD[site]) <= road_km + 3:
                return a["lat"], a["lon"], a.get("featureId") if layer == "gg25" else None
    return None


def compact(data):                                 # ne garde que 5 colonnes du registre
    keep = ("egid", "ganzwhg", "gkat", "gstat", "plz_plz6")
    return {"results": [{"featureId": f.get("featureId"),
                         "attributes": {k: f.get("attributes", {}).get(k) for k in keep},
                         "geometry": {"x": f.get("geometry", {}).get("x"),
                                      "y": f.get("geometry", {}).get("y")}}
                        for f in data.get("results", [])]}


def dwellings(search_value, field):
    """Bâtiments d'habitation (RegBL) : liste (lat, lon, nb logements, NPA). ~200 lignes par page."""
    rows, seen, offset = [], set(), 0
    while True:
        page = get(GEO + "MapServer/find",
                   {"layer": "ch.bfs.gebaeude_wohnungs_register", "searchText": str(search_value),
                    "searchField": field, "returnGeometry": "true", "sr": "4326",
                    "contains": "false", "offset": offset}, transform=compact)["results"]
        new = [f for f in page if f["featureId"] not in seen]
        if not new:
            break
        seen |= {f["featureId"] for f in new}
        rows += new
        offset += 200
        if len(page) < 200:
            break
    out, buildings = [], set()
    for f in rows:                                 # une ligne par logement -> une par bâtiment
        a, g = f["attributes"], f["geometry"]
        if g["x"] is None or a["egid"] in buildings or str(a.get("gstat")) not in ("1004", "None"):
            continue
        buildings.add(a["egid"])
        w = float(a.get("ganzwhg") or 0) or (1 if str(a.get("gkat")) in ("1020", "1030", "1040") else 0)
        if w > 0:
            out.append((g["y"], g["x"], w, str(a.get("plz_plz6", ""))[:4]))
    return out


def stops_near(lat, lon):
    r = get(OPENDATA + "locations", {"x": lat, "y": lon, "type": "station"})
    found = [s for s in r.get("stations", []) if s.get("id") and (s.get("coordinate") or {}).get("x")]
    found.sort(key=lambda s: s.get("distance") or 0)               # du plus proche au plus loin
    return [(s["id"], s["name"], s["coordinate"]["x"], s["coordinate"]["y"]) for s in found]


def wmean(values, weights):
    return sum(v * w for v, w in zip(values, weights)) / sum(weights)


def analyse_commune(commune, site, npa, road_km):
    # --- logements de la localité ---
    if commune in NPA_ONLY:                        # bâtiments du code postal des employés
        homes = dwellings(npa, "dplz4")            # (Lausanne 1004, Bramois 1967, Salins 1991)
    else:
        centre = commune_centre(commune, site, road_km)
        homes = dwellings(centre[2], "ggdenr")
        same_npa = [h for h in homes if h[3] == str(npa)]
        homes = same_npa if len(same_npa) >= 10 else homes
    lat, lon, w = [h[0] for h in homes], [h[1] for h in homes], [h[2] for h in homes]
    clat, clon = wmean(lat, w), wmean(lon, w)

    # --- arrêts candidats : autour du centre, du centre de chaque quart, + arrêt principal ---
    cands = {s[0]: s for s in stops_near(clat, clon)}
    mlat, mlon = sorted(lat)[len(lat) // 2], sorted(lon)[len(lon) // 2]
    for north in (True, False):
        for east in (True, False):
            q = [h for h in homes if (h[0] >= mlat) == north and (h[1] >= mlon) == east]
            if q:
                qlat, qlon = wmean([h[0] for h in q], [h[2] for h in q]), wmean([h[1] for h in q], [h[2] for h in q])
                for s in [s for s in stops_near(qlat, qlon) if s[0] not in cands][:5]:
                    cands[s[0]] = s
        # + arrêts proches des logements les plus au bord (N, S, E, O) : gares en limite de commune
    for h in (max(homes, key=lambda h: h[0]), min(homes, key=lambda h: h[0]),
              max(homes, key=lambda h: h[1]), min(homes, key=lambda h: h[1])):
        for s in [s for s in stops_near(h[0], h[1]) if s[0] not in cands][:5]:
            cands[s[0]] = s
    for name in [MAIN_STOP[commune]]:
        found = [(s["id"], s["name"], s["coordinate"]["x"], s["coordinate"]["y"])
                 for s in get(OPENDATA + "locations", {"query": name, "type": "station"}).get("stations", [])
                 if s.get("id") and (s.get("coordinate") or {}).get("x")]
        if found:
            cands.setdefault(found[0][0], found[0])
    cands = list(cands.values())

    # --- arrêts desservis et leur trajet vers le site (4 requêtes en parallèle) ---
    with ThreadPoolExecutor(4) as ex:
        served = [s for s, ok in zip(cands, ex.map(lambda s: served_6_9(s[0], s[1]), cands)) if ok]
        trips = dict(zip([s[0] for s in served], ex.map(lambda s: trip_0800(s[1], site), served)))
    served = [s for s in served if trips[s[0]]]

    # --- arrêt retenu : plus petit (marche moyenne + trajet) ; égalité -> marche la plus courte ---
    def walk(s):
        return wmean([km(h[0], h[1], s[2], s[3]) * 1000 * WALK_DETOUR / WALK_M_PER_MIN for h in homes], w)

    total = {s[0]: walk(s) + trip_minutes(trips[s[0]]) for s in served}
    best = min([s for s in served if total[s[0]] < min(total.values()) + TIE_MIN], key=walk)

    # part des logements pour qui cet arrêt serait aussi le meilleur choix individuel
    def own_best(h):
        return min(served, key=lambda s: km(h[0], h[1], s[2], s[3]) * 1000 * WALK_DETOUR / WALK_M_PER_MIN
                   + trip_minutes(trips[s[0]]))[0]
    share = sum(h[2] for h in homes if own_best(h) == best[0]) / sum(w)

    trip = trips[best[0]]
    return {
        "origin_stop": best[1],
        "n_dwellings": int(sum(w)),
        "share_dwellings_using_stop": round(share, 2),
        "dwellings_centre_latlon": f"{clat:.5f}, {clon:.5f}",
        "walk_to_origin_stop_min": round(walk(best), 1),
        "pt_stop_to_stop_min": trip_minutes(trip),
        "pt_transfers": trip.get("transfers"),
        "pt_total_via_origin_min": round(walk(best) + trip_minutes(trip), 1),
        "n_departures_6_9": departures_6_9(best[1], site),
        "arrival_before_0800": ts(trip["to"]["arrival"]).strftime("%H:%M"),
        "first_arrival": first_arrival(best[1], site),
    }

In [ ]:
# Chavannes : la gare de Renens est-elle candidate, et comment se compare-t-elle à Concorde ?
commune, site = "CHAVANNES PRES RENENS", "Lausanne"
r = communes[communes.commune_clean == commune].iloc[0]
centre = commune_centre(commune, site, r.mean_distance_km)
homes = [h for h in dwellings(centre[2], "ggdenr") if h[3] == str(r.postcode)]
w = [h[2] for h in homes]

# arrêts proches des 4 logements les plus au bord
bord = set()
for h in (max(homes, key=lambda h: h[0]), min(homes, key=lambda h: h[0]),
          max(homes, key=lambda h: h[1]), min(homes, key=lambda h: h[1])):
    bord |= {s[1] for s in stops_near(h[0], h[1])[:5]}
print("Gare de Renens trouvée par les bords :", "Renens VD" in bord)

# comparaison directe
for name in ["Chavannes-R., Concorde", "Renens VD"]:
    s = get(OPENDATA + "locations", {"query": name, "type": "station"})["stations"][0]
    marche = wmean([km(h[0], h[1], s["coordinate"]["x"], s["coordinate"]["y"]) * 1000
                    * WALK_DETOUR / WALK_M_PER_MIN for h in homes], w)
    trajet = trip_minutes(trip_0800(s["name"], site))
    print(f"{s['name']:<25} marche {marche:5.1f} + trajet {trajet:4.0f} = {marche + trajet:5.1f} min")

Gare de Renens trouvée par les bords : True
Chavannes-R., Concorde    marche   9.9 + trajet   12 =  21.9 min
Renens VD                 marche  16.3 + trajet    5 =  21.3 min


## 5. Calcul pour les 39 communes

⏱ Premier lancement : 15 à 45 minutes (téléchargement des horaires). Lancements suivants : quelques secondes grâce au cache.

In [ ]:
rows = []
try:
    for i, r in communes.iterrows():
        print(f"[{i + 1}/39] {r.commune_clean}", end=" ", flush=True)
        res = analyse_commune(r.commune_clean, r.site, r.postcode, r.mean_distance_km)
        print(f"-> {res['origin_stop']} ({res['pt_total_via_origin_min']} min)")
        rows.append({**r.to_dict(), **res})
finally:
    save_cache()

ta = pd.DataFrame(rows)

ta.head()

[1/39] ARBAZ -> Arbaz, poste (40.4 min)
[2/39] ARDON -> Ardon (23.3 min)
[3/39] AYENT -> Botyre (Ayent), Le Creux (46.0 min)
[4/39] BOTTENS -> Bottens, croisée (42.0 min)
[5/39] BRAMOIS -> Bramois, La Borgne (20.8 min)
[6/39] BUSSIGNY -> Bussigny, gare (20.6 min)
[7/39] CHAVANNES PRES RENENS -> Chavannes-R., Concorde (21.9 min)
[8/39] CHESEAUX SUR LAUSANNE -> Cheseaux-sur-L., gare (30.4 min)
[9/39] CONTHEY -> Conthey, bif. Conthey-Plan (22.5 min)
[10/39] CORCELLES LE JORAT -> Corcelles-le-Jorat, village (56.1 min)
[11/39] CRISSIER -> Crissier, Marcolet (23.5 min)
[12/39] CUGY -> Cugy VD, village (36.4 min)
[13/39] ECUBLENS -> Chavannes-R., Crochy (24.6 min)
[14/39] EPALINGES -> Epalinges, Croisettes (31.6 min)
[15/39] EVOLENE -> Evolène, Village (50.0 min)
[16/39] FROIDEVILLE -> Froideville, croisée (48.9 min)
[17/39] GRIMISUAT -> Grimisuat, Les Fermes (32.8 min)
[18/39] HEREMENCE -> Euseigne, Pont de Sauterot (62.5 min)
[19/39] LAUSANNE -> Lausanne, St-Paul (19.7 min)
[20/39] LE MONT 

,commune_clean,site,n_employees,postcode,mean_distance_km,origin_stop,n_dwellings,share_dwellings_using_stop,dwellings_centre_latlon,walk_to_origin_stop_min,pt_stop_to_stop_min,pt_transfers,pt_total_via_origin_min,n_departures_6_9,arrival_before_0800,first_arrival
0,ARBAZ,Sion,6,1974,11.483333,"Arbaz, poste",1095,0.24,"46.27767, 7.38430",12.4,28.0,1,40.4,4,07:50,06:47
1,ARDON,Sion,11,1957,13.809091,Ardon,1886,0.46,"46.20970, 7.25940",16.3,7.0,0,23.3,7,07:40,04:45
2,AYENT,Sion,15,1966,12.773333,"Botyre (Ayent), Le Creux",1943,0.06,"46.27941, 7.40969",19.0,27.0,0,46.0,6,07:48,06:30
3,BOTTENS,Lausanne,15,1041,14.440000,"Bottens, croisée",607,0.96,"46.61618, 6.66106",6.0,36.0,1,42.0,11,07:34,06:04
4,BRAMOIS,Sion,16,1967,5.318750,"Bramois, La Borgne",1954,0.25,"46.23140, 7.40063",10.8,10.0,0,20.8,19,07:51,06:05


### Temps voiture et ratio TP / voiture

Temps voiture = milieu de la fourchette Google + 5 min. Ratio = temps TP total (marche + trajet) ÷ temps voiture. Ratio < 1 : le TP est plus rapide que la voiture.

In [ ]:
# Temps voiture et ratio TP / voiture
ta["car_min_low"] = ta.commune_clean.map(lambda c: CAR_TIMES[c][0])
ta["car_min_high"] = ta.commune_clean.map(lambda c: CAR_TIMES[c][1])
ta["car_door_mid_min"] = (ta.car_min_low + ta.car_min_high) / 2 + CAR_TERMINAL_MIN
ta["ratio_pt_car_mid"] = (ta.pt_total_via_origin_min / ta.car_door_mid_min).round(2)
ta["ratio_pt_car_high"] = (ta.pt_total_via_origin_min / (ta.car_min_high + CAR_TERMINAL_MIN)).round(2)

ta[["site", "commune_clean", "pt_total_via_origin_min", "car_door_mid_min", "ratio_pt_car_mid"]]

,site,commune_clean,pt_total_via_origin_min,car_door_mid_min,ratio_pt_car_mid
0,Sion,ARBAZ,40.4,30.0,1.35
1,Sion,ARDON,23.3,23.0,1.01
2,Sion,AYENT,46.0,25.0,1.84
3,Lausanne,BOTTENS,42.0,39.5,1.06
4,Sion,BRAMOIS,20.8,15.5,1.34
5,Lausanne,BUSSIGNY,20.6,24.0,0.86
6,Lausanne,CHAVANNES PRES RENENS,21.9,21.0,1.04
7,Lausanne,CHESEAUX SUR LAUSANNE,30.4,31.5,0.97
8,Sion,CONTHEY,22.5,16.0,1.41
9,Lausanne,CORCELLES LE JORAT,56.1,43.0,1.30


## 6. Indice et groupes

- **Indice** : moyenne des 3 indicateurs standardisés (z-scores) : − ratio TP/voiture, log(1 + départs 6h-9h), − heure de première arrivée. Plus l'indice est élevé, meilleur est l'accès TP.
- **Groupes** : terciles de l'indice, pondérés par le nombre d'employés et **calculés dans chaque site**, pour que chaque groupe contienne des employés de Lausanne (traités) et de Sion (contrôle).

In [ ]:
fa = pd.to_datetime(ta.first_arrival, format="%H:%M")
first_min = fa.dt.hour * 60 + fa.dt.minute
z = lambda s: (s - s.mean()) / s.std()
index = lambda ratio: (z(-ratio) + z(np.log1p(ta.n_departures_6_9)) + z(-first_min)) / 3
ta["transit_index"] = index(ta.ratio_pt_car_mid).round(3)       # + = meilleur accès TP


def terciles(g):
    """3 groupes d'environ 1/3 des employés chacun, dans un site."""
    g = g.sort_values("transit_index")
    position = (g.n_employees.cumsum() - g.n_employees / 2) / g.n_employees.sum()
    return pd.cut(position, [0, 1 / 3, 2 / 3, 1], labels=["1_low", "2_medium", "3_high"]).astype(str)


ta["transit_group"] = pd.concat([terciles(g) for _, g in ta.groupby("site")])

### Export : `transit_access.csv` et `table_A1.csv`

In [ ]:
ta["source"] = (f"Horaires : transport.opendata.ch (horaires officiels CFF), jeudi {REF_DATE}. "
                "Logements : RegBL via api3.geo.admin.ch. Communes : swisstopo. "
                "Temps voiture : Google Maps, arrivée 08:00 un jeudi.")
ta["coding_notes"] = (
    "Arrêt retenu = arrêt desservi (>=1 départ 06-09h) qui minimise marche moyenne depuis tous les "
    "logements RegBL de la localité (vol d'oiseau x1.3, 80 m/min, pondéré par nb de logements) + trajet "
    "TP le plus rapide arrivant 07:30-08:00 ; égalité (<1 min) -> marche la plus courte. Candidats : "
    "arrêts proches du centre et du centre de chaque quart de la localité + arrêt principal "
    "Lausanne : logements du NPA 1004 ; Bramois, Salins : "
    "logements à moins de 1.5 km du village. Destination : gare principale du site. Services introduits "
    "après 2025 exclus (ligne 9 à Renens/Crissier, ligne 54, tram T1). Voiture : milieu de la fourchette "
    "Google + 5 min (stationnement + marche) ; ratio_pt_car_high = avec la borne haute. "
    "transit_index = moyenne des z-scores de (-ratio TP/voiture, log(1+départs 06-09h), -première "
    "arrivée). transit_group = terciles de l'indice pondérés par les employés, calculés dans chaque site. "
    "Corrections des données brutes : 'Savieze' -> SAVIESE ; NPA de Salins 1994 (= Aproz) -> 1991."
)

ta = ta.drop(columns="mean_distance_km")
ta.to_csv("transit_access.csv", index=False, encoding="utf-8-sig")

a1 = ta.sort_values(["site", "transit_index"], ascending=[True, False])[
    ["site", "commune_clean", "n_employees", "origin_stop", "pt_total_via_origin_min", "car_door_mid_min",
     "ratio_pt_car_mid", "n_departures_6_9", "first_arrival", "transit_index", "transit_group"]]
a1.to_csv("table_A1.csv", index=False, encoding="utf-8-sig")

a1

,site,commune_clean,n_employees,origin_stop,pt_total_via_origin_min,car_door_mid_min,ratio_pt_car_mid,n_departures_6_9,first_arrival,transit_index,transit_group
10,Lausanne,CRISSIER,45,"Crissier, Marcolet",23.5,27.0,0.87,49,04:59,1.261,3_high
18,Lausanne,LAUSANNE,138,"Lausanne, St-Paul",19.7,17.0,1.16,72,04:51,1.211,3_high
12,Lausanne,ECUBLENS,39,"Chavannes-R., Crochy",24.6,24.0,1.03,56,04:59,1.164,3_high
6,Lausanne,CHAVANNES PRES RENENS,20,"Chavannes-R., Concorde",21.9,21.0,1.04,47,04:59,1.085,3_high
27,Lausanne,RENENS,63,Renens VD,20.8,22.0,0.95,31,04:59,1.005,2_medium
23,Lausanne,MORGES,43,Morges,22.7,25.0,0.91,19,04:59,0.852,2_medium
25,Lausanne,PRILLY,43,Union-Prilly,20.7,17.5,1.18,28,04:51,0.819,2_medium
26,Lausanne,PULLY,49,Pully,16.8,16.5,1.02,25,05:11,0.752,2_medium
13,Lausanne,EPALINGES,29,"Epalinges, Croisettes",31.6,29.5,1.07,37,05:31,0.688,2_medium
20,Lausanne,LUTRY,46,"Lutry, gare",17.8,22.0,0.81,12,05:11,0.668,2_medium


## 7. Contrôles

Répartition des employés par groupe et robustesse au choix du temps voiture (borne haute de la fourchette Google).

In [ ]:
pd.set_option("display.width", 220)
print("\n", a1.to_string(index=False))
print("\nEmployés par groupe :\n", ta.pivot_table(index="site", columns="transit_group",
                                                   values="n_employees", aggfunc="sum"))
print("\nCorrélation de rang indice (voiture milieu) / indice (voiture max) :",
      round(ta.transit_index.corr(index(ta.ratio_pt_car_high), method="spearman"), 2))


     site         commune_clean  n_employees                    origin_stop  pt_total_via_origin_min  car_door_mid_min  ratio_pt_car_mid  n_departures_6_9 first_arrival  transit_index transit_group
Lausanne              CRISSIER           45             Crissier, Marcolet                     23.5              27.0              0.87                49         04:59          1.261        3_high
Lausanne              LAUSANNE          138              Lausanne, St-Paul                     19.7              17.0              1.16                72         04:51          1.211        3_high
Lausanne              ECUBLENS           39           Chavannes-R., Crochy                     24.6              24.0              1.03                56         04:59          1.164        3_high
Lausanne CHAVANNES PRES RENENS           20         Chavannes-R., Concorde                     21.9              21.0              1.04                47         04:59          1.085        3_high
Lausanne     

Le classement obtenu est plausible. Les communes proches des sites et desservies par le train ou le métro (Crissier, Lausanne, Ecublens, Chavannes-près-Renens ; Ardon, Conthey) obtiennent les scores les plus élevés, et les communes rurales ou de montagne (Ropraz, Savigny, Servion ; Hérémence, Saint-Martin, Ayent) les plus faibles, avec seulement 2 à 6 départs entre 6h et 9h pour les communes valaisannes. L’indice diminue avec la distance déclarée au site (corrélation de rang −0,51) et avec le temps de trajet en TP (−0,68). Il est robuste au choix du temps voiture : avec la borne haute de la fourchette Google, la corrélation de rang avec l’indice principal est de 0,99. Plusieurs limites doivent être soulignées. (1) Les horaires sont ceux du jeudi 10 décembre 2026 (horaire 2026), les horaires 2024-2025 n’étant plus disponibles ; les services introduits après l’enquête ont été exclus. (2) Chaque commune est représentée par un seul arrêt et par l’ensemble de ses logements ; dans les communes dispersées (Hérémence, Saint-Martin, Savièse, Savigny), la marche d’accès dépasse 20 minutes et est probablement surestimée, d’autant que le registre des bâtiments inclut des résidences secondaires. (3) L’adresse des sites étant inconnue, la destination est la gare principale. (4) Deux incohérences des données brutes ont été corrigées : l’orthographe de Savièse (« Savieze ») et le NPA de Salins (1994, celui d’Aproz, remplacé par 1991).